
# Add H5 files to database
## Single molecule tracking pipeline, python version

This script uploads the imaging data (in H5 format) to the storage and updates the database. Metadata will be processed automatically from info added during microscopy session. 

The experiment implies a single sample, all files in one folder; it might contain several FOVs, conditions, channels. To proceed, run the cells below by clicking the 'Play' button on the upper panel or select the cell and press Shift+Enter.


In [1]:
%load_ext dotenv
%dotenv config.env

In [2]:
%load_ext autoreload 
%autoreload 2

In [3]:
# Import the dependencies.

import os
import numpy as np
import pandas as pd
import h5py
import uuid
import tifffile

from matplotlib import pyplot as plt
from ipyfilechooser import FileChooser
from pathlib import Path
from datetime import datetime

from utils.set_widgets import im_file_chooser, filter_data
from utils.tiff_writer import create_array, write_tiff, h5_upload_img_allas
from utils.conditions_to_dict import conditions_to_df, h5_file_uuids 
from utils.csv_writer import update_db_img
from utils.set_widgets import experiment_widgets
from utils.conditions_to_dict import get_dict, read_h5_meta
from utils.xml_writer import get_xml_from_h5
from utils.csv_writer import h5_update_db_img



In [4]:
# Run this cell to chose the directory. 
# Click any file in it and press 'Select'.

path = Path(os.getenv('SMPP_INPUT_DIR'))
fc = FileChooser(path)
fc.filter_pattern = '*.h5'
display(fc)


FileChooser(path='/scratch/project_2009817/SMP_python_images', filename='', title='', show_hidden=False, selec…

In [5]:
# Check file sequence and experiment meta

exp_files = []

for i in Path(fc.selected_path).glob('*.h5'):
    f_dict = {}
    f_dict['path']=str(i)
    f_dict['name']=str(i).split('/')[-1]
    f_dict['led']=h5py.File(i, 'r')['MetaData'].attrs['LEDpower']
    f_dict['start']=h5py.File(i, 'r')['MetaData'].attrs['StartTime']
    f_dict['start_hr']=str(datetime.fromtimestamp(f_dict['start']).time())

    exp_files.append(f_dict)
    # print(i)
    # print(h5py.File(i, 'r')['MetaData'].attrs['LEDpower'], h5py.File(i, 'r')['MetaData'].attrs['StartTime'])



exp_files = sorted(exp_files, key=lambda d: d['start_hr'])
last = sorted(exp_files, key=lambda d: d['start_hr'])[-1]

for i in exp_files:
    print(f"{i['name']}, started: {i['start_hr']}, LED power: {i['led']}")

h5file = h5py.File(Path(fc.selected_path)/last['name'], 'r')
exp_name = '_'.join(last['name'].split('_')[:-1])
conditions = read_h5_meta(h5file)
conditions



261020_EN_Ec_64_M9glyCAAT_None_B.h5, started: 15:13:43.949446, LED power: 0.12
261020_EN_Ec_64_M9glyCAAT_None_C.h5, started: 15:14:13.672843, LED power: 0.0
261020_EN_Ec_64_M9glyCAAT_None_D.h5, started: 15:17:13.950571, LED power: 0.12
261020_EN_Ec_64_M9glyCAAT_None_E.h5, started: 15:17:45.354354, LED power: 0.0
261020_EN_Ec_64_M9glyCAAT_None_F.h5, started: 15:20:31.762708, LED power: 0.12
261020_EN_Ec_64_M9glyCAAT_None_G.h5, started: 15:20:59.195758, LED power: 0.0


{'name': 'Eveliny Nery',
 'date': '2026-10-20',
 'dye': 'None',
 'species': 'Escherichia coli',
 'strain': 'Ec_64',
 'temperature': '(np.float64(20.8), np.float64(20.94), np.float64(21.25))',
 'medium': 'M9glyCAAT',
 'chemical': 'None',
 'induction': '',
 'comment': '',
 'git_hash': '344a2bf621a3b7d66cd857806a7971c80d32e63d',
 'uuid_experiment': '4c06a400-cd11-11f1-9186-080038cf4ae9'}

In [6]:
# Create metadata XML 

xml_path = get_xml_from_h5(h5file, conditions, exp_name)


metadata xml created!


In [7]:
# Update the database (new experiment)

database = pd.read_csv(os.getenv('SMPP_DATABASE'), sep = '\t', dtype = 'str')
new_data = pd.DataFrame(columns = database.columns)
for i in new_data.columns:
    try:
        new_data[i] = [conditions[i]]
    except KeyError:
        pass
new_data['path'] = xml_path
database = pd.concat([database, new_data], ignore_index = True)
database.to_csv(os.getenv('SMPP_DATABASE'), sep = '\t', index = False)

print('database updated:')
database.tail(5)


database updated:


,name,date,dye,species,strain,temperature,medium,chemical,induction,comment,git_hash,uuid_experiment,uuid_fov,uuid_file,uuid_analysis,path
18398,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,261008_JB_Bs_02_other_JF552_E.h5,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,ed761928-cc86-11f1-91da-080038cf748b,ed7c9352-cc86-11f1-91da-080038cf748b,NaN,"('smt_1', '261008_JB_Bs_02_other_JF552_E.h5')"
18399,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,261008_JB_Bs_02_other_JF552_F.h5,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,ed761928-cc86-11f1-91da-080038cf748b,f09fc3e2-cc86-11f1-91da-080038cf748b,NaN,"('smt_1', '261008_JB_Bs_02_other_JF552_F.h5')"
18400,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,261008_JB_Bs_02_other_JF552_G.h5,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,f0eaabaa-cc86-11f1-91da-080038cf748b,f0f13754-cc86-11f1-91da-080038cf748b,NaN,"('smt_1', '261008_JB_Bs_02_other_JF552_G.h5')"
18401,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,261008_JB_Bs_02_other_JF552_H.h5,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,f0eaabaa-cc86-11f1-91da-080038cf748b,f708d660-cc86-11f1-91da-080038cf748b,NaN,"('smt_1', '261008_JB_Bs_02_other_JF552_H.h5')"
18402,Eveliny Nery,2026-10-20,None,Escherichia coli,Ec_64,"(np.float64(20.8), np.float64(20.94), np.float...",M9glyCAAT,None,,,344a2bf621a3b7d66cd857806a7971c80d32e63d,4c06a400-cd11-11f1-9186-080038cf4ae9,NaN,NaN,NaN,/projappl/project_2009817/Single-Molecule-Pipe...


In [8]:
# Upload the files, update the database.

fov_uuid = np.nan

for h5file in exp_files:
    # exp_files is list of dictionaries    

    # upload files to allas
    bucket_n, object_n = h5_upload_img_allas(fc.selected_path, h5file['name'])
    
    if h5file['led'] > 0:

        # assign FOV
        fov_uuid = str(uuid.uuid1()) 

        # get 
        file = h5py.File(Path(fc.selected_path)/h5file['name'], 'r')
        proj = np.mean(file['ImageData'], axis=0)
        wf_fname = h5file['name'].split('.')[0]+'_wf_proj.tif'
        tifffile.imwrite(Path(fc.selected_path)/wf_fname, proj) 
        print(f"widefield for omnipose is {wf_fname}")

    # identifiers for new h5 (fluo or second channel)
    git_hash, file_uuid, experiment_uuid = h5_file_uuids(new_data)
        
    # update the database
    updated = h5_update_db_img(new_data, h5file['name'], git_hash, file_uuid, fov_uuid, bucket_n, object_n)


updated.tail(20)



File uploaded to Allas, bucket: smt_1, object: 261020_EN_Ec_64_M9glyCAAT_None_B.h5
widefield for omnipose is 261020_EN_Ec_64_M9glyCAAT_None_B_wf_proj.tif
File uploaded to Allas, bucket: smt_1, object: 261020_EN_Ec_64_M9glyCAAT_None_C.h5
File uploaded to Allas, bucket: smt_1, object: 261020_EN_Ec_64_M9glyCAAT_None_D.h5
widefield for omnipose is 261020_EN_Ec_64_M9glyCAAT_None_D_wf_proj.tif
File uploaded to Allas, bucket: smt_1, object: 261020_EN_Ec_64_M9glyCAAT_None_E.h5
File uploaded to Allas, bucket: smt_1, object: 261020_EN_Ec_64_M9glyCAAT_None_F.h5
widefield for omnipose is 261020_EN_Ec_64_M9glyCAAT_None_F_wf_proj.tif
File uploaded to Allas, bucket: smt_1, object: 261020_EN_Ec_64_M9glyCAAT_None_G.h5


,name,date,dye,species,strain,temperature,medium,chemical,induction,comment,git_hash,uuid_experiment,uuid_fov,uuid_file,uuid_analysis,path
18389,Aakeel Wagay,2026-10-13,NaN,Escherichia coli,Ec_64,"(np.float64(30.0), np.float64(36.99), np.float...",M9glyCAAT,NaN,NaN,FOV 2_30,0ca957412b19d722725763bc0db2e14170a4527a,f6312ce4-cc5c-11f1-a14e-080038cf7449,fcc378b4-cc5c-11f1-a14e-080038cf7449,fcd15038-cc5c-11f1-a14e-080038cf7449,6926b600-cc5e-11f1-a79c-080038cf7449,/projappl/project_2009817/Single-Molecule-Pipe...
18390,Aakeel Wagay,2026-10-13,NaN,Escherichia coli,Ec_64,"(np.float64(30.0), np.float64(36.99), np.float...",M9glyCAAT,NaN,NaN,FOV 3_37_5-7mins during hs,0ca957412b19d722725763bc0db2e14170a4527a,f6312ce4-cc5c-11f1-a14e-080038cf7449,feebd0a0-cc5c-11f1-a14e-080038cf7449,fef980b0-cc5c-11f1-a14e-080038cf7449,8d2d07fc-cc5e-11f1-a79c-080038cf7449,/projappl/project_2009817/Single-Molecule-Pipe...
18391,Aakeel Wagay,2026-10-13,NaN,Escherichia coli,Ec_64,"(np.float64(30.0), np.float64(36.99), np.float...",M9glyCAAT,NaN,NaN,FOV 4_37_10-12mins during hs,0ca957412b19d722725763bc0db2e14170a4527a,f6312ce4-cc5c-11f1-a14e-080038cf7449,00f4e3c8-cc5d-11f1-a14e-080038cf7449,0103d43c-cc5d-11f1-a14e-080038cf7449,9b9ae43a-cc5e-11f1-a79c-080038cf7449,/projappl/project_2009817/Single-Molecule-Pipe...
18392,Aakeel Wagay,2026-10-13,NaN,Escherichia coli,Ec_64,"(np.float64(30.0), np.float64(36.99), np.float...",M9glyCAAT,NaN,NaN,FOV 5_37_15-17mins during hs,0ca957412b19d722725763bc0db2e14170a4527a,f6312ce4-cc5c-11f1-a14e-080038cf7449,030a4144-cc5d-11f1-a14e-080038cf7449,0319dc6c-cc5d-11f1-a14e-080038cf7449,a862810a-cc5e-11f1-a79c-080038cf7449,/projappl/project_2009817/Single-Molecule-Pipe...
18393,Aakeel Wagay,2026-10-13,NaN,Escherichia coli,Ec_64,"(np.float64(30.0), np.float64(36.99), np.float...",M9glyCAAT,NaN,NaN,FOV 6_37_20-22mins during hs,0ca957412b19d722725763bc0db2e14170a4527a,f6312ce4-cc5c-11f1-a14e-080038cf7449,056655a4-cc5d-11f1-a14e-080038cf7449,057505a4-cc5d-11f1-a14e-080038cf7449,b63c9dce-cc5e-11f1-a79c-080038cf7449,/projappl/project_2009817/Single-Molecule-Pipe...
18394,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,S750 25uM IPTG,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,NaN,NaN,NaN,/projappl/project_2009817/Single-Molecule-Pipe...
18395,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,261008_JB_Bs_02_other_JF552_B.h5,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,e9772092-cc86-11f1-91da-080038cf748b,e99e19ae-cc86-11f1-91da-080038cf748b,NaN,"('smt_1', '261008_JB_Bs_02_other_JF552_B.h5')"
18396,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,261008_JB_Bs_02_other_JF552_C.h5,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,e9e97fd4-cc86-11f1-91da-080038cf748b,e9f25550-cc86-11f1-91da-080038cf748b,NaN,"('smt_1', '261008_JB_Bs_02_other_JF552_C.h5')"
18397,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,261008_JB_Bs_02_other_JF552_D.h5,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,e9e97fd4-cc86-11f1-91da-080038cf748b,ed1231f6-cc86-11f1-91da-080038cf748b,NaN,"('smt_1', '261008_JB_Bs_02_other_JF552_D.h5')"
18398,Jessie Bullock,2026-10-08,JF552,Bacillus subtilis,Bs_02,"(np.float64(36.99), np.float64(37.0), np.float...",other,IPTG,NaN,261008_JB_Bs_02_other_JF552_E.h5,0ca957412b19d722725763bc0db2e14170a4527a,d09f32ee-cc86-11f1-91da-080038cf748b,ed761928-cc86-11f1-91da-080038cf748b,ed7c9352-cc86-11f1-91da-080038cf748b,NaN,"('smt_1', '261008_JB_Bs_02_other_JF552_E.h5')"


In [16]:
# To process a new experiment, press 'Kernel' -> 'Restart Kernel'. 
# Otherwise, you can close the add_imaging_data notebook, or shut down the interactive session.
